In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
dataset=pd.read_csv('/content/used_cars.csv')

In [19]:
dataset.shape

(4009, 12)

In [20]:

dataset.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [21]:
dataset.isnull().sum()

,0
brand,0
model,0
model_year,0
milage,0
fuel_type,170
engine,0
transmission,0
ext_col,0
int_col,0
accident,113


In [22]:
import pandas as pd

# Make sure the dataset variable actually exists and is loaded
try:
    dataset
except NameError:
    dataset = pd.read_csv("used_cars.csv")

categorical_cols = ['brand', 'model', 'fuel_type', 'transmission',
                     'ext_col', 'int_col', 'accident', 'clean_title']

# Only convert columns that actually exist in the dataframe
missing_cols = [col for col in categorical_cols if col not in dataset.columns]
if missing_cols:
    print("Warning: these columns were not found and will be skipped:", missing_cols)

categorical_cols = [col for col in categorical_cols if col in dataset.columns]

for col in categorical_cols:
    dataset[col] = dataset[col].astype('category')

print("Categorical columns converted:")
print(dataset[categorical_cols].dtypes)

Categorical columns converted:
brand           category
model           category
fuel_type       category
transmission    category
ext_col         category
int_col         category
accident        category
clean_title     category
dtype: object


In [23]:
import numpy as np

dataset['car_age'] = 2026 - dataset['model_year']
dataset['horsepower'] = dataset['engine'].str.extract(r'([\d\.]+)\s*HP').astype(float)
dataset['displacement_L'] = dataset['engine'].str.extract(r'([\d\.]+)\s*L\b').astype(float)

print("New features created:")
dataset[['car_age', 'horsepower', 'displacement_L']].head()

New features created:


,car_age,horsepower,displacement_L
0,13,300.0,3.7
1,5,NaN,3.8
2,4,NaN,NaN
3,11,354.0,3.5
4,5,NaN,2.0


In [24]:
import numpy as np

# Clean price first — remove $ and commas, convert to numeric
dataset['price'] = dataset['price'].astype(str).str.replace(r'[\$,]', '', regex=True)
dataset['price'] = pd.to_numeric(dataset['price'], errors='coerce')

# Now log-transform is safe
dataset['log_price'] = np.log1p(dataset['price'])

print("Price skew before:", dataset['price'].skew())
print("Log price skew after:", dataset['log_price'].skew())

Price skew before: 19.51355120421606
Log price skew after: 0.1057263535507364


In [25]:
feature_cols = categorical_cols + ['milage', 'car_age', 'horsepower', 'displacement_L']

X = dataset[feature_cols]
y = dataset['log_price']

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (4009, 12)
Target shape: (4009,)


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (3207, 12)
Test shape: (802, 12)


In [27]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 15,
    'max_depth': 6,
    'min_data_in_leaf': 20,
    'max_bin': 63,
    'verbose': -1,
    'force_col_wise': True,
    'num_threads': 2,
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert 'milage' to numeric in X_train and X_test
# Remove ' mi.' and ',' then convert to float
X_train['milage'] = X_train['milage'].astype(str).str.replace(' mi.', '', regex=False).str.replace(',', '', regex=False).astype(float)
X_test['milage'] = X_test['milage'].astype(str).str.replace(' mi.', '', regex=False).str.replace(',', '', regex=False).astype(float)

train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols)
test_data = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_cols, reference=train_data)

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)]
)

Training until validation scores don't improve for 30 rounds
[50]	train's rmse: 0.25541	test's rmse: 0.361305
[100]	train's rmse: 0.219192	test's rmse: 0.35213
[150]	train's rmse: 0.197453	test's rmse: 0.349094
[200]	train's rmse: 0.181913	test's rmse: 0.347758
[250]	train's rmse: 0.169191	test's rmse: 0.34613
Early stopping, best iteration is:
[262]	train's rmse: 0.166774	test's rmse: 0.34597


In [28]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import time


if dataset['milage'].dtype == 'object':
    dataset['milage'] = dataset['milage'].astype(str).str.replace(' mi.', '', regex=False)
    dataset['milage'] = dataset['milage'].str.replace(',', '', regex=False)
    dataset['milage'] = pd.to_numeric(dataset['milage'], errors='coerce')


if dataset['price'].dtype == 'object':
    dataset['price'] = dataset['price'].astype(str).str.replace(r'[\$,]', '', regex=True)
    dataset['price'] = pd.to_numeric(dataset['price'], errors='coerce')

dataset['log_price'] = np.log1p(dataset['price'])


if 'car_age' not in dataset.columns:
    dataset['car_age'] = 2026 - dataset['model_year']
if 'horsepower' not in dataset.columns:
    dataset['horsepower'] = dataset['engine'].str.extract(r'([\d\.]+)\s*HP').astype(float)
if 'displacement_L' not in dataset.columns:
    dataset['displacement_L'] = dataset['engine'].str.extract(r'([\d\.]+)\s*L\b').astype(float)


model_freq = dataset['model'].value_counts(normalize=True)
dataset['model_freq'] = dataset['model'].map(model_freq)


numeric_check = ['milage', 'car_age', 'horsepower', 'displacement_L', 'model_freq']
for col in numeric_check:
    dataset[col] = pd.to_numeric(dataset[col], errors='coerce')
print("Numeric dtypes check:")
print(dataset[numeric_check].dtypes)


categorical_cols = ['brand', 'fuel_type', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title']
feature_cols = categorical_cols + numeric_check

X = dataset[feature_cols].copy()
y = dataset['log_price']

for col in categorical_cols:
    X[col] = X[col].astype('category')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_cols)
test_data = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_cols, reference=train_data)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 15,
    'max_depth': 6,
    'min_data_in_leaf': 20,
    'max_bin': 63,
    'verbose': -1,
    'force_col_wise': True,
    'num_threads': 2,
}

print("Starting training...")
start = time.time()

model = lgb.train(
    params,
    train_data,
    num_boost_round=100,
    valid_sets=[train_data, test_data],
    valid_names=['train', 'test'],
    callbacks=[lgb.early_stopping(15), lgb.log_evaluation(10)]
)

print(f"Training finished in {time.time() - start:.1f} seconds.")

Numeric dtypes check:
milage              int64
car_age             int64
horsepower        float64
displacement_L    float64
model_freq        float64
dtype: object
Starting training...
Training until validation scores don't improve for 15 rounds
[10]	train's rmse: 0.467198	test's rmse: 0.528702
[20]	train's rmse: 0.343403	test's rmse: 0.422204
[30]	train's rmse: 0.294443	test's rmse: 0.385858
[40]	train's rmse: 0.269797	test's rmse: 0.370044
[50]	train's rmse: 0.255131	test's rmse: 0.361286
[60]	train's rmse: 0.244933	test's rmse: 0.357707
[70]	train's rmse: 0.236084	test's rmse: 0.354337
[80]	train's rmse: 0.228294	test's rmse: 0.35214
[90]	train's rmse: 0.221514	test's rmse: 0.350322
[100]	train's rmse: 0.215279	test's rmse: 0.349993
Did not meet early stopping. Best iteration is:
[97]	train's rmse: 0.217109	test's rmse: 0.349847
Training finished in 0.1 seconds.


In [29]:
importance = pd.DataFrame({
    'feature': model.feature_name(),
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print(importance.head(10))

           feature   importance
7           milage  5203.851711
9       horsepower  1985.940677
8          car_age  1948.095603
0            brand   973.061522
10  displacement_L   674.432695
11      model_freq    96.666253
2     transmission    81.964600
1        fuel_type    48.819252
4          int_col    34.341963
3          ext_col    18.517458


In [30]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
r2 = r2_score(y_test_orig, y_pred)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test R-squared: {r2:.4f}")

Test RMSE: 131862.1322
Test R-squared: 0.1493


In [32]:
import pandas as pd
import numpy as np

def predict_car_price(
    brand,
    model_name,
    model_year,
    milage_str,
    fuel_type,
    engine_str,
    transmission,
    ext_col,
    int_col,
    accident,
    clean_title
):
    input_data = pd.DataFrame([{
        'brand': brand,
        'model': model_name,
        'model_year': model_year,
        'milage': milage_str,
        'fuel_type': fuel_type,
        'engine': engine_str,
        'transmission': transmission,
        'ext_col': ext_col,
        'int_col': int_col,
        'accident': accident,
        'clean_title': clean_title
    }])

    input_data['milage'] = input_data['milage'].astype(str).str.replace(' mi.', '', regex=False).str.replace(',', '', regex=False).astype(float)

    input_data['car_age'] = 2026 - input_data['model_year']
    input_data['horsepower'] = input_data['engine'].str.extract(r'([\d\.]+)\s*HP').astype(float)
    input_data['displacement_L'] = input_data['engine'].str.extract(r'([\d\.]+)\s*L\b').astype(float)

    input_data['model_freq'] = input_data['model'].map(model_freq).fillna(model_freq.mean())

    input_data['horsepower'] = input_data['horsepower'].fillna(0)
    input_data['displacement_L'] = input_data['displacement_L'].fillna(0)

    categorical_cols_model = ['brand', 'fuel_type', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title']

    for col in categorical_cols_model:
        if col in dataset.columns and isinstance(dataset[col].dtype, pd.CategoricalDtype):
            input_data[col] = pd.Categorical(input_data[col], categories=dataset[col].cat.categories)
        else:
            input_data[col] = input_data[col].astype('category')

    final_features_for_model = ['brand', 'fuel_type', 'transmission', 'ext_col', 'int_col',
                                'accident', 'clean_title', 'milage', 'car_age', 'horsepower',
                                'displacement_L', 'model_freq']

    X_predict = input_data[final_features_for_model]

    log_predicted_price = model.predict(X_predict)

    predicted_price = np.expm1(log_predicted_price)[0]

    return predicted_price


predicted = predict_car_price(
    brand='Ford',
    model_name='Utility Police Interceptor Base',
    model_year=2013,
    milage_str='51,000 mi.',
    fuel_type='E85 Flex Fuel',
    engine_str='300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...',
    transmission='6-Speed A/T',
    ext_col='Black',
    int_col='Black',
    accident='At least 1 accident or damage reported',
    clean_title='Yes'
)
print(f"Predicted Car Price: ${predicted:.2f}")

Predicted Car Price: $18614.63


In [34]:
import joblib

# Save the LightGBM model
model_filename = 'lightgbm_car_price_model.joblib'
joblib.dump(model, model_filename)
print(f"Model saved to {model_filename}")

Model saved to lightgbm_car_price_model.joblib


In [35]:
import joblib
import pickle

# Load the model saved by joblib to confirm it works
loaded_model = joblib.load('lightgbm_car_price_model.joblib')
print("Model loaded successfully using joblib.")

# Now save it explicitly as a .pkl file using the pickle module
pkl_model_filename = 'lightgbm_car_price_model.pkl'
with open(pkl_model_filename, 'wb') as file:
    pickle.dump(loaded_model, file)

print(f"Model also saved as a .pkl file to {pkl_model_filename}")

Model loaded successfully using joblib.
Model also saved as a .pkl file to lightgbm_car_price_model.pkl
